# 11. One-shot confirmatory holdout: BTCUSDT spot, 2018-01-01 to 2020-12-31

All my earlier results (notebooks 01–10) are exploratory: I obtained them while repeatedly inspecting 2023–2026. Here I test the pipeline (price-based 4% events, V+F score, alert budgets) exactly once on 2018–2020 Binance spot BTCUSDT one-minute data that the pipeline has never evaluated, with the already frozen model, thresholds and policies and with hypotheses written down in advance. Futures one-minute data exist only from 2019-09, so I examine a cross-venue candidate only on 2019-11 to 2020-12.

- This test is confirmatory and one-shot. I freeze the hypotheses, thresholds and candidates in `HOLDOUT_SPEC.json`, with its SHA-256, before loading the holdout data, and I evaluate the holdout once. I report the results as observed, favorable or not.
- I use the frozen outputs of notebooks 01–07 and the candidate files of notebooks 08 (cross-venue information) and 09 (capacity design).
- The spectral-feature (hS) versus volatility check in my earlier work used 2018-01 to 2020-09 once, with a different event definition; this is the first use with this pipeline.
- I train the model on 2021 and apply it to the earlier years 2018–2020 (a *backcast*); 2018 had a different trading environment.
- I write the results to `btc_lead_cost_results_77months_spot/HOLDOUT_2018_2020_confirmatory_v1/`.

In [2]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
# --- emit_policy, as in notebook 06 ---
def emit_policy_nb6(t, scores, threshold, mode, *, credits=None, B=240, month_cap=20,
                burst=20, cooldown=240):
    t=np.asarray(t,np.int64);sc=np.asarray(scores,float)
    assert len(t)==len(sc) and (len(t)<2 or (np.diff(t)>0).all())
    if mode=='risk_bucket':
        assert credits is not None and len(credits)==len(t)
        credits=np.asarray(credits,float)
    assert mode in {'calendar','constant_bucket','risk_bucket'}
    cand=np.flatnonzero(np.isfinite(sc)&(sc>=threshold))
    emitted=[];annual=0;monthly=0;py=None;pm=None;prev=-10**25
    tokens=float(burst); last_t=None;last_credit=None
    for k in cand:
        tm=int(t[k]);stamp=pd.Timestamp(tm,tz='UTC')
        yr=int(stamp.year);mon=int(stamp.month)
        if yr!=py:
            py=yr;annual=0;tokens=float(burst);last_t=tm
            last_credit=None if credits is None else float(credits[k])
        if (yr,mon)!=pm:
            pm=(yr,mon);monthly=0
        if mode=='constant_bucket':
            if last_t is not None and tm>last_t:
                tokens=min(float(burst),tokens+(tm-last_t)/NS*B/(365.2425*1440))
            last_t=tm
        elif mode=='risk_bucket':
            if last_credit is not None:
                tokens=min(float(burst),tokens+max(0,float(credits[k])-last_credit))
            last_credit=float(credits[k])
        if annual>=B:continue
        if mode=='calendar' and monthly>=month_cap:continue
        if mode!='calendar' and tokens<1-1e-9:continue
        if tm-prev<cooldown*NS:continue
        emitted.append(tm);prev=tm;annual+=1;monthly+=1
        if mode!='calendar':tokens=max(0.0,tokens-1.0)
    return np.array(emitted,dtype=np.int64)


RISK_MIN = 0.5; RISK_MAX = 2.0; TRAILING_DAYS = 7


def daily_risk_weights(idx, close, volume, ref=None):
    """Notebook 06 causal daily risk weight (trailing 7 complete UTC days of 1m squared returns / 2021 median),
    plus the budget-neutral variant: divided by its own trailing 365-day mean (previous days only), re-clipped."""
    logclose = np.log(close)
    ret = np.r_[np.nan, np.diff(logclose)]
    usable = (np.isfinite(ret) & np.isfinite(close) & (close > 0) & (volume > 0) &
              np.r_[False, (volume[:-1] > 0) & np.isfinite(close[:-1]) & (close[:-1] > 0)])
    ret2 = pd.Series(np.where(usable, ret * ret, np.nan), index=idx)
    D = pd.DataFrame({'ms': ret2.resample('D').mean(), 'n': ret2.resample('D').count()})
    D.loc[D.n < 1000, 'ms'] = np.nan
    D['trailing_7d_rv'] = np.sqrt(D.ms.shift(1).rolling(TRAILING_DAYS, min_periods=TRAILING_DAYS).mean())
    if ref is None:
        ref = D.loc['2021-01-01':'2021-12-31', 'trailing_7d_rv'].median()
    assert np.isfinite(ref) and ref > 0
    D['risk_weight'] = np.clip(D.trailing_7d_rv / ref, RISK_MIN, RISK_MAX).fillna(1.0)
    norm = D.risk_weight.shift(1).rolling(365, min_periods=90).mean()
    D['risk_weight_bn'] = np.clip(D.risk_weight / norm, RISK_MIN, RISK_MAX).fillna(1.0)
    return D, float(ref)


def credits(idx, daily_weight):
    w = daily_weight.reindex(idx.floor('D')).to_numpy(float)
    assert np.isfinite(w).all()
    return np.cumsum(B / (365.2425 * 1440) * w, dtype=np.float64)


def calibrate_bucket(t_cal, s_cal, mode, credits_cal=None, grid=None):
    """Notebook 06 rule: closest to 240 alerts in 2022, strictest threshold on ties."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        a = emit_policy_nb6(t_cal, s_cal, float(th), mode, credits=credits_cal if mode == 'risk_bucket' else None)
        rows.append((float(th), len(a), abs(len(a) - B)))
    df = pd.DataFrame(rows, columns=['threshold', 'calibration_alarms', 'distance_to_240'])
    row = df.sort_values(['distance_to_240', 'threshold'], ascending=[True, False]).iloc[0]
    return float(row.threshold), int(row.calibration_alarms), df
MODES = ['calendar', 'rolling30', 'paced', 'token_bucket']
# --- time_features, as in notebook 04 ---
def time_features(ts_ns):
    idx=pd.to_datetime(np.asarray(ts_ns,dtype='int64'),utc=True)
    months=(idx.year.to_numpy()*12+idx.month.to_numpy()).astype('int32')
    years=idx.year.to_numpy(dtype='int32')
    naive=idx.tz_localize(None)
    first=naive.to_period('M').to_timestamp()
    after=first+pd.offsets.MonthBegin(1)
    frac=np.asarray((naive-first)/pd.Timedelta(minutes=1),dtype=float) / np.asarray((after-first)/pd.Timedelta(minutes=1),dtype=float)
    return months,years,frac

# --- token_capacity, as in notebook 04 ---
def token_capacity(B): return max(2,int(math.ceil(B/48)))

# --- emit_policy, as in notebook 04 ---
def emit_policy_nb4_py(t, scores, thresh, B, mode, burst=None, detailed=False):
    """At each contemporaneous score crossing, decide to emit or discard; no event label used."""
    t=np.asarray(t,dtype='int64');scores=np.asarray(scores,dtype='float64')
    assert len(t)==len(scores) and np.all(np.diff(t)>0)
    assert mode in MODES and B>0 and B%12==0
    mon,yr,frac=time_features(t)
    cap=B//12
    if burst is None:burst=token_capacity(B)
    if burst<1:raise ValueError('bucket capacity must be >=1')
    # Continuous refill: nominal B tokens per 365 days, initial full bucket.
    refill_per_min=B/(365.0*24*60)
    candidate=np.flatnonzero(np.isfinite(scores)&(scores>=float(thresh)))
    out=[];prev_alert=-10**21;prev_year=-1;prev_month=-1
    count_year=0;count_month=0
    rolling=deque();tokens=float(burst);last_token_time=int(t[0]) if len(t) else 0
    detail=defaultdict(lambda:{'score_crossings':0,'alerts':0,'annual_blocks':0,'monthly_blocks':0,
          'pacing_blocks':0,'rolling_blocks':0,'token_blocks':0,'cooldown_blocks':0,
          'exhausted_utc':None,'first7_alerts':0})
    for i in candidate:
        ti=int(t[i]);mi=int(mon[i]);yi=int(yr[i]);dt=detail[mi]
        dt['score_crossings']+=1
        if yi!=prev_year:
            prev_year=yi;count_year=0
        if mi!=prev_month:
            prev_month=mi;count_month=0
        if mode=='token_bucket':
            elapsed=max(0,(ti-last_token_time)/NS)
            tokens=min(float(burst),tokens+elapsed*refill_per_min)
            last_token_time=ti
        if count_year>=B:
            dt['annual_blocks']+=1;continue
        if mode=='calendar' or mode=='paced':
            if count_month>=cap:
                dt['monthly_blocks']+=1;continue
            if mode=='paced':
                allowance=min(cap, 2+int(np.floor(cap*float(frac[i]))))
                if count_month>=allowance:
                    dt['pacing_blocks']+=1;continue
        elif mode=='rolling30':
            while rolling and ti-rolling[0]>=30*24*60*NS:rolling.popleft()
            if len(rolling)>=cap:
                dt['rolling_blocks']+=1;continue
        elif mode=='token_bucket':
            if tokens<1.0-1e-10:
                dt['token_blocks']+=1;continue
        if ti-prev_alert<COOLDOWN_MIN*NS:
            dt['cooldown_blocks']+=1;continue
        out.append(ti);prev_alert=ti;count_year+=1;count_month+=1;dt['alerts']+=1
        if frac[i] < 7/31: dt['first7_alerts']+=1  # approximate; exact first7 available below
        if mode=='rolling30':rolling.append(ti)
        if mode=='token_bucket':tokens=max(0.,tokens-1.)
        if mode in ('calendar','paced') and count_month==cap and dt['exhausted_utc'] is None:
            dt['exhausted_utc']=ti
    out=np.asarray(out,dtype='int64')
    if detailed:return out,dict(detail)
    return out


_MODE_CODE = {'calendar': 0, 'rolling30': 1, 'paced': 2, 'token_bucket': 3}


@njit(cache=False)
def _emit_core_nb4(t, scores, thresh, B_, mode_code, burst, cap, frac, months, years, cooldown_ns, refill_per_min, ns, roll_ns):
    # Port of the notebook 04 schedulers (verified identical to the pure-Python scheduler).
    n = len(t)
    out = np.empty(n, np.int64); k = 0; head = 0
    prev_alert = -(2**62); prev_year = -1; prev_month = -1
    count_year = 0; count_month = 0
    tokens = float(burst)
    last_token_time = t[0] if n > 0 else 0
    for i in range(n):
        s = scores[i]
        if not (np.isfinite(s) and s >= thresh):
            continue
        ti = t[i]; mi = months[i]; yi = years[i]
        if yi != prev_year:
            prev_year = yi; count_year = 0
        if mi != prev_month:
            prev_month = mi; count_month = 0
        if mode_code == 3:
            elapsed = max(0.0, (ti - last_token_time) / ns)
            tokens = min(float(burst), tokens + elapsed * refill_per_min)
            last_token_time = ti
        if count_year >= B_:
            continue
        if mode_code == 0 or mode_code == 2:
            if count_month >= cap:
                continue
            if mode_code == 2:
                allowance = min(cap, 2 + int(np.floor(cap * frac[i])))
                if count_month >= allowance:
                    continue
        elif mode_code == 1:
            while head < k and ti - out[head] >= roll_ns:
                head += 1
            if k - head >= cap:
                continue
        elif mode_code == 3:
            if tokens < 1.0 - 1e-10:
                continue
        if ti - prev_alert < cooldown_ns:
            continue
        out[k] = ti; k += 1; prev_alert = ti; count_year += 1; count_month += 1
        if mode_code == 3:
            tokens = max(0.0, tokens - 1.0)
    return out[:k].copy()


def emit_policy_nb4(t, scores, thresh, B_, mode, burst=None):
    t = np.asarray(t, dtype='int64'); scores = np.asarray(scores, dtype='float64')
    assert len(t) == len(scores) and np.all(np.diff(t) > 0)
    assert mode in MODES and B_ > 0 and B_ % 12 == 0
    if burst is None:
        burst = token_capacity(B_)
    mon, yr, frac = time_features(t)
    return _emit_core_nb4(t, scores, float(thresh), int(B_), _MODE_CODE[mode], int(burst), int(B_ // 12),
                          np.asarray(frac, np.float64), np.asarray(mon, np.int64), np.asarray(yr, np.int64),
                          np.int64(COOLDOWN_MIN * NS), B_ / (365.0 * 24 * 60), np.int64(NS), np.int64(30 * 24 * 60 * NS))
SURGE_BASE_CAP = 16                        # regular alerts per month (80% of 20)
SURGE_RESERVE = B - 12 * SURGE_BASE_CAP    # 48 alerts per UTC year usable only during a crash cluster
SURGE_ACCRUAL = SURGE_RESERVE // 12        # the reserve accrues 4 per month, so by the end of month m at most 20*m alerts


def make_surge_flags(confirmed_ns, surge_days):
    conf = np.sort(np.asarray(confirmed_ns, np.int64))

    def flags(t):
        j = np.searchsorted(conf, t, 'left') - 1          # last crash confirmed strictly before minute t
        out = np.zeros(len(t), bool); ok = j >= 0
        out[ok] = (t[ok] - conf[j[ok]]) < surge_days * 1440 * NS
        return out
    return flags


@njit(cache=False)
def _reserve_core(t, elig, surge, years, months, B_, base_cap, reserve, accrual, cooldown_ns):
    out = np.empty(len(t), np.int64); kind = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; rused = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; rused = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B_ or ti - last < cooldown_ns:
            continue
        if mcount < base_cap:
            mcount += 1; kind[k] = 0
        elif surge[i] and rused < min(reserve, accrual * (mo - yr * 12)):
            rused += 1; kind[k] = 1
        else:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1
    return out[:k].copy(), kind[:k].copy()


def emit_reserve_calendar(t, eligible, surge, base_cap=SURGE_BASE_CAP, reserve=SURGE_RESERVE, B_=240, cooldown_min=240):
    """Calendar policy with a base monthly quota plus a reserve that accrues reserve/12 per month within the UTC year
    and can only be spent after the month quota is used and while a crash was confirmed in the previous surge_days.
    Cumulative alerts never exceed the 20/month calendar pace. Returns (alert times, kinds 0=base 1=reserve)."""
    t = np.asarray(t, np.int64)
    assert len(t) < 2 or np.all(np.diff(t) > 0)
    years, months = cal_fields(t)
    return _reserve_core(t, np.asarray(eligible, bool), np.asarray(surge, bool), years, months, int(B_), int(base_cap),
                         int(reserve), int(reserve // 12), np.int64(cooldown_min * NS))


def emit_reserve_calendar_py(t, eligible, surge, base_cap=SURGE_BASE_CAP, reserve=SURGE_RESERVE, B_=240, cooldown_min=240):
    """Plain-Python reference used only to check the compiled version."""
    out = []; kinds = []; mcount = ycount = rused = 0; pm = py = None; last = -(10**20)
    for i in np.flatnonzero(eligible):
        ti = int(t[i]); d = pd.Timestamp(ti, tz='UTC'); yr = d.year; mo = yr * 12 + d.month
        if yr != py:
            ycount = 0; rused = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B_ or ti - last < cooldown_min * NS:
            continue
        if mcount < base_cap:
            mcount += 1; kinds.append(0)
        elif surge[i] and rused < min(reserve, (reserve // 12) * d.month):
            rused += 1; kinds.append(1)
        else:
            continue
        out.append(ti); last = ti; ycount += 1
    return np.asarray(out, np.int64), np.asarray(kinds, np.int64)


def check_reserve(t, s, th, surge):
    m = in_period(t, '2022-06-01', '2023-01-01') if (t >= ts('2022-06-01').value).any() else np.ones(len(t), bool)
    a, ka = emit_reserve_calendar(t[m], s[m] >= th, surge[m]); b, kb = emit_reserve_calendar_py(t[m], s[m] >= th, surge[m])
    if not (np.array_equal(a, b) and np.array_equal(ka, kb)):
        raise AssertionError('compiled reserve calendar differs from the Python reference')
    return len(a)
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

XV_WIN = 15                      # minutes, pre-declared
ZWIN = 30 * 1440; ZMIN = 15 * 1440  # same trailing 30-day z-score window as z_vol / z_hs
XV_FEATURES = ['z_basis', 'd_flow', 'z_vr']


def zscore_trailing(s):
    r = s.rolling(ZWIN, min_periods=ZMIN)
    return (s - r.mean().shift(1)) / r.std().shift(1).replace(0, np.nan)


def cross_venue_features(spot_px, fut_px):
    """Causal cross-venue features at minute t (both frames: regular UTC 1m grid, columns close/volume/taker_buy_vol).
    z_basis: 15-min mean of log(futures/spot) in bps, z-scored against the previous 30 days.
    d_flow : 15-min taker-sell share on futures minus the same share on spot.
    z_vr   : log(futures/spot 15-min volume), z-scored against the previous 30 days."""
    assert spot_px.index.equals(fut_px.index)
    sc = spot_px.close.where(spot_px.volume > 0); fc = fut_px.close.where(fut_px.volume > 0)
    basis = np.log(fc / sc) * 1e4
    z_basis = zscore_trailing(basis.rolling(XV_WIN, min_periods=XV_WIN).mean())
    sv = spot_px.volume.rolling(XV_WIN, min_periods=XV_WIN).sum()
    fv = fut_px.volume.rolling(XV_WIN, min_periods=XV_WIN).sum()
    ss = (spot_px.volume - spot_px.taker_buy_vol).rolling(XV_WIN, min_periods=XV_WIN).sum()
    fs = (fut_px.volume - fut_px.taker_buy_vol).rolling(XV_WIN, min_periods=XV_WIN).sum()
    d_flow = fs / fv.replace(0, np.nan) - ss / sv.replace(0, np.nan)
    z_vr = zscore_trailing(np.log(fv.replace(0, np.nan) / sv.replace(0, np.nan)))
    return pd.DataFrame({'z_basis': z_basis.astype('float32'), 'd_flow': d_flow.astype('float32'),
                         'z_vr': z_vr.astype('float32')}, index=spot_px.index)


FEATURE_SETS = {
    'M0_VF': ['z_vol', 'tsr'],
    'M1_VF_basis': ['z_vol', 'tsr', 'z_basis'],
    'M2_VF_flowgap': ['z_vol', 'tsr', 'd_flow'],
    'M3_VF_volratio': ['z_vol', 'tsr', 'z_vr'],
    'M4_VF_all3': ['z_vol', 'tsr', 'z_basis', 'd_flow', 'z_vr'],
}

HOLD = SPOT_BASE / 'HOLDOUT_2018_2020_confirmatory_v1'
HOLD.mkdir(parents=True, exist_ok=True)
RAW_SPOT = DRIVE / 'BTCUSDT_spot_holdout_2017_2020' / '1m'
RAW_FUT = DRIVE / 'BTCUSDT_futures_holdout_2019_2020' / '1m'
H_START, H_END, XV_START = '2018-01-01', '2021-01-01', '2019-11-01'
SPOT_MONTHS = month_list('2017-08', '2020-12'); FUT_MONTHS = month_list('2019-09', '2020-12')
N_SHIFTS = 10000; SEED = 20261029; ALPHA = 0.05
LOCK = HOLD / 'HOLDOUT_EVALUATED.lock'
if LOCK.exists():
    raise RuntimeError(f'This holdout has already been evaluated (one-shot). Results: {HOLD}. It is not evaluated again.')

XV_CAND = SPOT_BASE / 'crossvenue_information_v1' / 'crossvenue_candidate.json'
CAP_CAND = SPOT_BASE / 'capacity_design_v1' / 'capacity_candidate.json'
NB4_CAL = SPOT_BASE / 'systems_engineering_scheduler_oracle_value_v1' / 'threshold_calibration_2022.csv'
for p in (XV_CAND, CAP_CAND, NB4_CAL):
    if not p.is_file():
        raise FileNotFoundError(f'Results of a prerequisite notebook are missing: {p}')
xv_c = json.loads(XV_CAND.read_text()); cap_c = json.loads(CAP_CAND.read_text())
nb4cal = pd.read_csv(NB4_CAL)
TH4 = {r.mode: float(r.threshold_2022) for r in nb4cal[nb4cal.B == 240].itertuples()}
XV_SEL = xv_c.get('selected'); CAP_SEL = cap_c.get('selected')
CANDIDATES = [c for c in [('capacity', CAP_SEL), ('crossvenue', XV_SEL)] if c[1]]

HSPEC = {
    'title': 'One-shot confirmatory holdout, BTCUSDT spot 1m, 2018-01-01 to 2021-01-01 (exclusive)',
    'data': {'spot': 'Binance spot BTCUSDT monthly 1m archives 2017-08..2020-12, SHA-256 verified',
             'futures': 'Binance USD-M BTCUSDT monthly 1m 2019-09..2020-12 (only if a cross-venue candidate exists)'},
    'prior_use': 'not evaluated by this pipeline; GitHub hS-vs-volatility study used 2018-01..2020-09 once with a different event definition',
    'model': 'V+F logistic, 2021 spot cache (identical to notebook 5), applied backwards (backcast)',
    'events': 'notebook-2 price-only rule (4% below prior 240-min peak, 5 closes, 24h refractory) on the downloaded series; '
              'evaluation events: breach in [2018-01-01 04:00, 2021-01-01)',
    'primary_endpoint': 'alert 30-240 min before the 4% breach AND before the 1% onset (pre-onset hit), one alert per event',
    'frozen_thresholds': {'TH_CAL': TH_CAL, 'TH_HIGH': TH_HIGH, 'TH_BUCKET': TH_BUCKET, 'notebook4_B240': TH4},
    'H1_skill': {'claim': 'frozen calendar V+F (20/month, 240/yr) catches more pre-onset events than its own alerts shifted in time',
                 'test': f'one-sided circular-shift permutation, {N_SHIFTS} shifts of 7 days to (period - 7 days), seed {SEED}',
                 'alpha': ALPHA},
    'H2_scheduler': {'claim': 'calendar cap beats rolling30, paced and token_bucket (notebook-4 designs, B=240, 2022 thresholds, warm-up from 2017-12-01)',
                     'test': 'one-sided exact McNemar on pre-onset hits, Holm over 3 comparisons', 'alpha': ALPHA},
    'H3_capacity': 'descriptive: share of calendar-missed events with no monthly room left during their whole warning window; signal-reachable events',
    'H4_candidates': {'candidates': CANDIDATES, 'capacity_candidate': cap_c, 'crossvenue_candidate': xv_c,
                      'test': 'one-sided exact McNemar (candidate better) on pre-onset hits vs the matching calendar baseline, Holm over candidates',
                      'crossvenue_window': [XV_START, H_END],
                      'crossvenue_not_evaluable_if': 'downloaded futures 2020 do not match the futures cache exactly, or cross-venue features do not match'},
    'secondary': 'high-score calendar and constant bucket (notebooks 5-6 frozen thresholds); any-hit endpoint; year-by-year',
    'guards': ['downloaded 2020 spot prices equal the spot cache', 'recomputed 2020-06..12 features equal the cache',
               'recomputed 2020-03..12 events equal the spot catalog', 'model refit reproduces notebook-5 threshold',
               'candidate thresholds reproduce notebooks 8-9'],
    'source_sha256': {p.name: sha256_file(p) for p in (XV_CAND, CAP_CAND, NB4_CAL, REF_PATH, THR_PATH)},
    'rules': 'all outcomes reported as observed; no rule changes after reading outcomes; second evaluation refused (lock file)'}
raw_spec = (json.dumps(HSPEC, indent=2, sort_keys=True, default=float) + '\n').encode()
spec_path = HOLD / 'HOLDOUT_SPEC.json'
if spec_path.exists() and spec_path.read_bytes() != raw_spec:
    raise RuntimeError('Differs from the existing HOLDOUT_SPEC.json. The frozen specification cannot be changed for a new evaluation.')
spec_path.write_bytes(raw_spec)
(HOLD / 'HOLDOUT_SPEC.sha256').write_text(hashlib.sha256(raw_spec).hexdigest() + '  HOLDOUT_SPEC.json\n')
print('Holdout specification frozen before loading holdout data. SHA-256:', hashlib.sha256(raw_spec).hexdigest())
print('Candidates:', CANDIDATES or 'none (H4 not tested)')

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Holdout specification frozen before loading holdout data. SHA-256: 5e759b32b6d01c83a13ed826f4c48f6c11d8a6d7e47a1409aa60979c32887caa
Candidates: none (H4 not tested)


## 1. Holdout data, 2018–2020

In [3]:
man_s = get_archives('spot', 'BTCUSDT', SPOT_MONTHS, RAW_SPOT)
man_s.to_csv(HOLD / 'spot_download_manifest.csv', index=False)
man_f = None
if XV_SEL:
    man_f = get_archives('futures/um', 'BTCUSDT', FUT_MONTHS, RAW_FUT, allow_missing=True)
    man_f.to_csv(HOLD / 'futures_download_manifest.csv', index=False)

BTCUSDT spot {'downloaded': 41}


## 2. Recompute features and the event catalog

In [4]:
H = regularize(load_archives(man_s, RAW_SPOT), end=pd.Timestamp('2020-12-31 23:59', tz='UTC'))
H = H.loc[:pd.Timestamp('2020-12-31 23:59', tz='UTC')]
FH = compute_spot_features(H)
px, sg = load_btc_spot()
ov = H.index[(H.index >= ts('2020-06-01')) & (H.index < ts('2021-01-01'))]
for col in ['close', 'volume', 'taker_buy_vol']:
    a = H.loc[ov, col].to_numpy(float); b = px.loc[ov, col].to_numpy(float)
    both = np.isfinite(a) & np.isfinite(b)
    if (np.isfinite(a) != np.isfinite(b)).any() or not np.array_equal(a[both], b[both]):
        raise RuntimeError(f'GUARD FAIL: downloaded 2020 spot {col} differs from the spot cache')
for col, tol in [('z_vol', 5e-4), ('tsr', 5e-6), ('z_hs', 5e-4)]:
    a = FH.loc[ov, col].to_numpy(float); b = sg.loc[ov, col].to_numpy(float)
    both = np.isfinite(a) & np.isfinite(b)
    if (np.isfinite(a) != np.isfinite(b)).mean() > 1e-4 or both.sum() < 100_000:
        raise RuntimeError(f'GUARD FAIL: {col} validity pattern differs from the cache')
    md = float(np.max(np.abs(a[both] - b[both])))
    print(f'{col}: max |recomputed - cache| on 2020-06..12 = {md:.3g}')
    if md > tol:
        raise RuntimeError(f'GUARD FAIL: {col} differs from the cache by {md}')
Eh = price_only_catalog(H.close.to_numpy(float), H.volume.to_numpy(float), H.index)
Ec = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')


def _win(E_):
    return E_[(E_.breach_utc >= ts('2020-03-01')) & (E_.breach_utc < ts('2020-12-31 20:00'))].reset_index(drop=True)


ea, eb = _win(Eh), _win(Ec)
if len(ea) != len(eb) or any(not np.array_equal(ea[c].astype('int64').to_numpy(), eb[c].astype('int64').to_numpy())
                             for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']):
    raise RuntimeError(f'GUARD FAIL: recomputed 2020 events differ from the spot catalog ({len(ea)} vs {len(eb)})')
print('PASS: prices, features and', len(ea), 'events in the 2020 overlap match the existing spot pipeline')
EV_H = events_between(Eh, H_START, H_END)
print('Holdout evaluation events:', len(EV_H), '| by year', EV_H.groupby(EV_H.breach_utc.dt.year).size().to_dict())

FFT computation 100,031/1,775,280
FFT computation 1,775,280/1,775,280
z_vol: max |recomputed - cache| on 2020-06..12 = 0
tsr: max |recomputed - cache| on 2020-06..12 = 0
z_hs: max |recomputed - cache| on 2020-06..12 = 0
PASS: prices, features and 38 events in the 2020 overlap match the existing spot pipeline
Holdout evaluation events: 199 | by year {2018: 106, 2019: 51, 2020: 42}


## 3. Model refit (2021 cache) and holdout scores

In [5]:
T_c = px.index.asi8
close_c = px.close.to_numpy(float); volume_c = px.volume.to_numpy(float)
v_c = sg.z_vol.to_numpy(float); f_c = sg.tsr.to_numpy(float); h_c = sg.z_hs.to_numpy(float)
valid_c = (np.isfinite(close_c) & (close_c > 0) & np.isfinite(volume_c) & (volume_c > 0) & np.isfinite(v_c) & np.isfinite(f_c) & np.isfinite(h_c) & (f_c >= 0) & (f_c <= 1))
y_c = label_from_catalog(T_c, Ec)
tr_c = train_mask(T_c, valid_c)
model = new_logit().fit(np.column_stack([v_c, f_c])[tr_c], y_c[tr_c])
cal_c = valid_c & in_period(T_c, '2022-01-01', '2023-01-01')
S_c = np.full(len(T_c), np.nan); S_c[valid_c] = model.predict_proba(np.column_stack([v_c, f_c])[valid_c])[:, 1]
th_chk, _, _ = calibrate_calendar(T_c[cal_c], S_c[cal_c])
if abs(th_chk - TH_CAL) > 1e-12:
    raise RuntimeError(f'GUARD FAIL: refit threshold {th_chk} != notebook 5 {TH_CAL}')
print('PASS: notebook-5 model and 2022 threshold reproduced')

T_h = H.index.asi8
close_h = H.close.to_numpy(float); volume_h = H.volume.to_numpy(float)
v_h = FH.z_vol.to_numpy(float); f_h = FH.tsr.to_numpy(float); h_h = FH.z_hs.to_numpy(float)
valid_h = (np.isfinite(close_h) & (close_h > 0) & np.isfinite(volume_h) & (volume_h > 0) & np.isfinite(v_h) & np.isfinite(f_h) & np.isfinite(h_h) & (f_h >= 0) & (f_h <= 1))
S_h = np.full(len(T_h), np.nan); S_h[valid_h] = model.predict_proba(np.column_stack([v_h, f_h])[valid_h])[:, 1]
hm = valid_h & in_period(T_h, H_START, H_END)
T_H, S_H = T_h[hm], S_h[hm]
print(f'holdout scored minutes: {len(T_H):,}; share of scores above TH_CAL: {(S_H >= TH_CAL).mean():.4f}')

PASS: notebook-5 model and 2022 threshold reproduced
holdout scored minutes: 1,569,222; share of scores above TH_CAL: 0.0641


## 4. Generate alerts with the frozen policies

In [6]:
POL = {}
POL['calendar'] = emit_calendar_fast(T_H, S_H >= TH_CAL)
if not np.array_equal(POL['calendar'], emit_calendar(T_H, S_H >= TH_CAL)):
    raise RuntimeError('fast calendar differs from the reference implementation')
POL['high_score'] = emit_calendar_fast(T_H, S_H >= TH_HIGH)
POL['constant_bucket'] = emit_policy_nb6(T_H, S_H, TH_BUCKET, 'constant_bucket')

wm = valid_h & in_period(T_h, '2017-12-01', H_END)
T_W, S_W = T_h[wm], S_h[wm]
chk = in_period(T_W, '2017-12-01', '2018-03-01')
SCHED = {}
for mode in MODES:
    if not np.array_equal(emit_policy_nb4(T_W[chk], S_W[chk], TH4[mode], 240, mode),
                          emit_policy_nb4_py(T_W[chk], S_W[chk], TH4[mode], 240, mode)):
        raise RuntimeError(f'fast notebook 04 scheduler differs from the reference ({mode})')
    a = emit_policy_nb4(T_W, S_W, TH4[mode], 240, mode)
    SCHED[mode] = a[a >= ts(H_START).value]

if CAP_SEL == 'P3_surge_reserve_calendar':
    prm = cap_c['parameters']
    flags_h = make_surge_flags(Eh.confirmed_utc.astype('int64').to_numpy(), int(prm['surge_days']))
    check_reserve(T_H, S_H, float(prm['threshold']), flags_h(T_H))
    POL['capacity_candidate'], _ = emit_reserve_calendar(T_H, S_H >= float(prm['threshold']), flags_h(T_H),
                                                         base_cap=int(prm['base_cap']), reserve=int(prm['reserve']))
elif CAP_SEL == 'P2_risk_bucket_budget_neutral':
    prm = cap_c['parameters']
    D_c, ref_c = daily_risk_weights(px.index, close_c, volume_c)
    if abs(ref_c - float(prm['risk_ref_2021'])) > 1e-15:
        raise RuntimeError('GUARD FAIL: 2021 risk reference differs from notebook 9')
    C_c = credits(px.index, D_c.risk_weight_bn)
    th_bn, _, _ = calibrate_bucket(T_c[cal_c], S_c[cal_c], 'risk_bucket', credits_cal=C_c[cal_c])
    if abs(th_bn - float(prm['threshold'])) > 1e-12:
        raise RuntimeError('GUARD FAIL: P2 threshold differs from notebook 9')
    D_h, _ = daily_risk_weights(H.index, close_h, volume_h, ref=ref_c)
    C_h = credits(H.index, D_h.risk_weight_bn)
    POL['capacity_candidate'] = emit_policy_nb6(T_H, S_H, th_bn, 'risk_bucket', credits=C_h[hm])

XV_RESULT = None
if XV_SEL:
    reason = None
    Fh = regularize(load_archives(man_f, RAW_FUT), end=pd.Timestamp('2020-12-31 23:59', tz='UTC')).reindex(H.index)
    futc = pd.read_parquet(FUT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet', columns=['close', 'volume', 'taker_buy_vol']).sort_index()
    futc.index = futc.index.tz_convert('UTC'); futc = futc.reindex(px.index)
    o20 = H.index[(H.index >= ts('2020-01-01')) & (H.index < ts('2021-01-01'))]
    for col in ['close', 'volume', 'taker_buy_vol']:
        a = Fh.loc[o20, col].to_numpy(float); b = futc.loc[o20, col].to_numpy(float)
        both = np.isfinite(a) & np.isfinite(b)
        if both.sum() < 0.99 * len(o20) or not np.isclose(a[both], b[both], rtol=2e-8, atol=1e-7).all():
            reason = f'downloaded 2020 futures {col} do not match the futures cache'
            break
    if reason is None:
        XVh = cross_venue_features(H[['close', 'volume', 'taker_buy_vol']], Fh)
        XVc = cross_venue_features(px[['close', 'volume', 'taker_buy_vol']], futc)
        for col, tol in [('z_basis', 1e-3), ('d_flow', 1e-6), ('z_vr', 1e-3)]:
            a = XVh.loc[ov, col].to_numpy(float); b = XVc.loc[ov, col].to_numpy(float)
            both = np.isfinite(a) & np.isfinite(b)
            if both.sum() < 100_000 or float(np.max(np.abs(a[both] - b[both]))) > tol:
                reason = f'cross-venue feature {col} differs between download and cache'
                break
    if reason is None:
        FEATc = {'z_vol': v_c, 'tsr': f_c, **{c: XVc[c].to_numpy(float) for c in XV_FEATURES}}
        FEATh = {'z_vol': v_h, 'tsr': f_h, **{c: XVh[c].to_numpy(float) for c in XV_FEATURES}}
        common_c = valid_c & np.isfinite(XVc[XV_FEATURES].to_numpy(float)).all(axis=1)
        common_h = valid_h & np.isfinite(XVh[XV_FEATURES].to_numpy(float)).all(axis=1)
        trc = train_mask(T_c, common_c); calc = common_c & in_period(T_c, '2022-01-01', '2023-01-01')
        xw = common_h & in_period(T_h, XV_START, H_END)
        out = {}
        for tag, cols, th_exp in [('baseline_M0_VF', ['z_vol', 'tsr'], xv_c['baseline_common_mask']['threshold_2022']),
                                  ('candidate_' + XV_SEL, xv_c['features'], xv_c['threshold_2022'])]:
            Xc = np.column_stack([FEATc[c] for c in cols]); Xh = np.column_stack([FEATh[c] for c in cols])
            m = new_logit().fit(Xc[trc], y_c[trc])
            th, _, _ = calibrate_calendar(T_c[calc], m.predict_proba(Xc[calc])[:, 1])
            if abs(th - float(th_exp)) > 1e-12:
                reason = f'{tag}: threshold {th} differs from notebook 8 ({th_exp})'
                break
            out[tag] = emit_calendar_fast(T_h[xw], m.predict_proba(Xh[xw])[:, 1] >= th)
        if reason is None:
            XV_RESULT = out
    if reason:
        print('Cross-venue candidate not evaluable (pre-declared rule):', reason)
    XV_REASON = reason
print('alerts per policy:', {k: len(a) for k, a in POL.items()}, '| schedulers:', {k: len(a) for k, a in SCHED.items()})

alerts per policy: {'calendar': 713, 'high_score': 708, 'constant_bucket': 704} | schedulers: {'calendar': 713, 'rolling30': 709, 'paced': 704, 'token_bucket': 704}


## 5. Preregistered hypothesis tests (where I first see the outcomes)

In [7]:
def holm(ps):
    order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * ps[i])); adj[i] = run
    return adj


summary = {}
# H1
a = np.sort(POL['calendar']); obs = statistics(EV_H, a)
t0 = ts(H_START).value; L = (ts(H_END).value - t0) // NS
rng = np.random.default_rng(SEED)
deltas = rng.integers(7 * 1440, L - 7 * 1440, size=N_SHIFTS)
null_pre = np.empty(N_SHIFTS, int); null_hit = np.empty(N_SHIFTS, int)
for i, dl in enumerate(deltas):
    sh = np.sort(t0 + (a - t0 + int(dl) * NS) % (L * NS))
    m_ = match_events(EV_H, sh); null_hit[i] = len(m_); null_pre[i] = int(m_.pre1pct.sum())
p_h1 = (1 + int((null_pre >= obs['before_1pct']).sum())) / (N_SHIFTS + 1)
p_h1_any = (1 + int((null_hit >= obs['caught']).sum())) / (N_SHIFTS + 1)
summary['H1'] = dict(events=len(EV_H), alarms=obs['alarms'], pre_onset_hits=obs['before_1pct'], any_hits=obs['caught'],
                     null_mean_pre_onset=float(null_pre.mean()), null_p95_pre_onset=float(np.percentile(null_pre, 95)),
                     p_one_sided=p_h1, p_any_hit_secondary=p_h1_any, supported=bool(p_h1 < ALPHA))
pd.DataFrame({'shift_minutes': deltas, 'pre_onset_hits': null_pre, 'any_hits': null_hit}).to_csv(HOLD / 'H1_circular_shift_null.csv', index=False)

# H2
rows = []
for mode in ['rolling30', 'paced', 'token_bucket']:
    p = paired(EV_H, SCHED['calendar'], SCHED[mode])
    p_cal_better = mcnemar_one_sided(p['new_only_pre1pct'], p['base_only_pre1pct'])
    rows.append(dict(comparison=f'calendar_vs_{mode}', calendar_alarms=len(SCHED['calendar']), other_alarms=len(SCHED[mode]),
                     calendar_pre1pct=p['base_pre1pct'], other_pre1pct=p['new_pre1pct'],
                     calendar_only=p['base_only_pre1pct'], other_only=p['new_only_pre1pct'], p_one_sided=p_cal_better))
h2 = pd.DataFrame(rows); h2['p_holm'] = holm(h2.p_one_sided.to_numpy()); h2['supported'] = h2.p_holm < ALPHA
h2.to_csv(HOLD / 'H2_scheduler_tests.csv', index=False)
summary['H2'] = h2.to_dict('records')

# H3
am = pd.to_datetime(a, utc=True).tz_localize(None).to_period('M').astype(str).to_numpy()
hit = set(match_events(EV_H, a).event_id); dec = []
for e in EV_H.itertuples(index=False):
    b = int(e.breach_utc.value); room = False
    for ti in range(b - LEAD_MAX * NS, b - LEAD_MIN * NS + 1, NS):
        mo = str(pd.Timestamp(ti, tz='UTC').tz_localize(None).to_period('M'))
        if int((am[:np.searchsorted(a, ti, 'right')] == mo).sum()) < MONTH_CAP:
            room = True; break
    w = (T_H >= b - LEAD_MAX * NS) & (T_H <= b - LEAD_MIN * NS)
    dec.append(dict(event_id=int(e.event_id), breach_utc=e.breach_utc,
                    category='caught' if int(e.event_id) in hit else ('missed_no_room' if not room else 'missed_some_room'),
                    signal_reachable=bool((S_H[w] >= TH_CAL).any())))
dec = pd.DataFrame(dec); dec.to_csv(HOLD / 'H3_capacity_decomposition.csv', index=False)
cnt = dec.category.value_counts().to_dict(); missed = len(dec) - cnt.get('caught', 0)
summary['H3'] = dict(**cnt, share_of_misses_with_no_room=(cnt.get('missed_no_room', 0) / missed) if missed else None,
                     signal_reachable=int(dec.signal_reachable.sum()))

# H4
h4 = []
if CAP_SEL:
    p = paired(EV_H, POL['calendar'], POL['capacity_candidate'])
    h4.append(dict(candidate=CAP_SEL, window=f'{H_START}..{H_END}', base_alarms=len(POL['calendar']), cand_alarms=len(POL['capacity_candidate']),
                   base_pre1pct=p['base_pre1pct'], cand_pre1pct=p['new_pre1pct'], base_only=p['base_only_pre1pct'], cand_only=p['new_only_pre1pct'],
                   p_one_sided=p['p_pre1pct_one_sided_new_better'], evaluable=True))
if XV_SEL:
    if XV_RESULT is None:
        h4.append(dict(candidate=XV_SEL, window=f'{XV_START}..{H_END}', evaluable=False, note=XV_REASON, p_one_sided=np.nan))
    else:
        ev_x = events_between(Eh, XV_START, H_END)
        kb, kc = list(XV_RESULT)
        p = paired(ev_x, XV_RESULT[kb], XV_RESULT[kc])
        h4.append(dict(candidate=XV_SEL, window=f'{XV_START}..{H_END}', events=len(ev_x), base_alarms=len(XV_RESULT[kb]), cand_alarms=len(XV_RESULT[kc]),
                       base_pre1pct=p['base_pre1pct'], cand_pre1pct=p['new_pre1pct'], base_only=p['base_only_pre1pct'], cand_only=p['new_only_pre1pct'],
                       p_one_sided=p['p_pre1pct_one_sided_new_better'], evaluable=True))
h4 = pd.DataFrame(h4)
if len(h4):
    ev_ok = h4.evaluable.astype(bool).to_numpy()
    h4['p_holm'] = np.nan
    if ev_ok.any():
        h4.loc[ev_ok, 'p_holm'] = holm(h4.loc[ev_ok, 'p_one_sided'].to_numpy(float))
    h4['supported'] = h4.p_holm < ALPHA
h4.to_csv(HOLD / 'H4_candidate_tests.csv', index=False)
summary['H4'] = h4.to_dict('records')

# secondary
sec = []
for k, al in {**POL, **{f'sched_{m}': SCHED[m] for m in MODES}}.items():
    st = statistics(EV_H, al); sec.append(dict(policy=k, **st))
    for year in (2018, 2019, 2020):
        ev_y = EV_H[EV_H.breach_utc.dt.year == year]; a_y = al[pd.to_datetime(al, utc=True).year == year]
        if len(ev_y):
            s_ = statistics(ev_y, a_y); sec.append(dict(policy=k, year=year, alarms=len(a_y), caught=s_['caught'], before_1pct=s_['before_1pct']))
    pd.DataFrame({'alarm_utc': pd.to_datetime(al, utc=True)}).to_csv(HOLD / f'alarms_{k}.csv', index=False)
sec = pd.DataFrame(sec); sec.to_csv(HOLD / 'secondary_policy_results.csv', index=False)
EV_H.to_csv(HOLD / 'holdout_events.csv', index=False)

(HOLD / 'HOLDOUT_RESULTS.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
LOCK.write_text(json.dumps({'evaluated_utc': pd.Timestamp.now(tz='UTC').isoformat(), 'spec_sha256': hashlib.sha256(raw_spec).hexdigest()}))
print('H1 skill:', summary['H1'])
print('\nH2 scheduler:'); display(h2.round(4))
print('\nH3 capacity:', summary['H3'])
print('\nH4 candidates:'); display(h4.round(4) if len(h4) else 'no candidates were selected by notebooks 8-9')
print('\nSecondary (all events 2018-2020, then by year):'); display(sec.round(4))
print('\nLocked: results reported as observed.')

H1 skill: {'events': 199, 'alarms': 713, 'pre_onset_hits': 13, 'any_hits': 36, 'null_mean_pre_onset': 11.3724, 'null_p95_pre_onset': 17.0, 'p_one_sided': 0.3531646835316468, 'p_any_hit_secondary': 9.999000099990002e-05, 'supported': False}

H2 scheduler:


,comparison,calendar_alarms,other_alarms,calendar_pre1pct,other_pre1pct,calendar_only,other_only,p_one_sided,p_holm,supported
0,calendar_vs_rolling30,713,709,13,14,8,9,0.6855,1.0000,False
1,calendar_vs_paced,713,704,13,13,6,6,0.6128,1.0000,False
2,calendar_vs_token_bucket,713,704,13,9,7,3,0.1719,0.5156,False



H3 capacity: {'missed_no_room': 103, 'missed_some_room': 60, 'caught': 36, 'share_of_misses_with_no_room': 0.6319018404907976, 'signal_reachable': 100}

H4 candidates:


'no candidates were selected by notebooks 8-9'


Secondary (all events 2018-2020, then by year):


,policy,alarms,caught,before_1pct,false_alarms,recall,pre_onset_recall,alert_precision,year
0,calendar,713,36,13,677.0,0.1809,0.0653,0.0505,NaN
1,calendar,233,18,5,NaN,NaN,NaN,NaN,2018.0
2,calendar,240,10,5,NaN,NaN,NaN,NaN,2019.0
3,calendar,240,8,3,NaN,NaN,NaN,NaN,2020.0
4,high_score,708,36,14,672.0,0.1809,0.0704,0.0508,NaN
5,high_score,228,19,5,NaN,NaN,NaN,NaN,2018.0
6,high_score,240,9,5,NaN,NaN,NaN,NaN,2019.0
7,high_score,240,8,4,NaN,NaN,NaN,NaN,2020.0
8,constant_bucket,704,40,15,664.0,0.2010,0.0754,0.0568,NaN
9,constant_bucket,224,22,6,NaN,NaN,NaN,NaN,2018.0



Locked: results reported as observed.
